# Boltz Inference Guide

**Goal**: Test whether Boltz structural confidence scores (pTM, ipTM, pLDDT) for predicted Cas–crRNA–target ternary complexes correlate with experimentally observed guide RNA activity across multiple Cas proteins and datasets.

This mirrors the AlphaFold3 experiment already run on NSCC (TIGER × CasRx, 500 guides, 5 seeds) and extends it to six datasets spanning CasRx, DjCas13d, SpCas9, and Cas12a.

---

## Contents
1. [Boltz vs AF3 — What Changes](#section-1)
2. [Datasets Overview](#section-2)
3. [Protein Sequences](#section-3)
4. [Target Sequence Reconstruction](#section-4)
5. [Boltz YAML Input Format](#section-5)
6. [PBS Scripts for NSCC](#section-6)
7. [Parsing Boltz Outputs](#section-7)
8. [Statistical Analysis](#section-8)
9. [Pre-flight Checklist](#section-9)

---
## 1. Boltz vs AF3 — What Changes

| Concern | AlphaFold3 | Boltz-2 |
|---|---|---|
| Input format | JSON (`dialect: alphafold3`) | YAML |
| RNA alphabet | U (existing JSONs already use U) | U — same |
| Protein MSA | Separate CPU-only PBS job (8 h, 128 GB RAM) | Single GPU command; no separate MSA phase |
| Seeds | `modelSeeds: [1,2,3,4,5]` inside the JSON | `--diffusion_samples 5` CLI flag |
| Inference runtime | Singularity image (`alphafold3.sif`) | pip-installed conda env — **no Singularity** |
| Weights | ~3 × 100 GB `.bin.zst` files, pre-loaded on NSCC | `boltz2_conf.ckpt` (2.29 GB) + `mols.tar` (1.86 GB), transferred manually via scp |
| PBS job count | 4 (fill → MSA → inference → analyze) | 3 (install → inference → analyze) |
| Confidence JSON name | `*summary_confidences_0.json` | `*confidence_model_*.json` (different keys — confirm on NSCC) |
| pLDDT scale | 0–100 | 0–1 (multiply by 100 before comparison) |

**Key consequence**: Boltz-2 collapses the two-phase AF3 pipeline (MSA + inference) into a single GPU call. The PBS array is simpler: one script per dataset.

> **Weight files**: `pip install boltz` installs the **code only**. The two required weight files must be downloaded from HuggingFace and transferred to NSCC scratch manually — see Section 6.  
> **`boltz2_aff.ckpt`** (2.06 GB, also in the HuggingFace repo) is for affinity prediction only — **do not download it**, it is not needed for structure prediction.

---
## 2. Datasets Overview

Four confirmed datasets based on files available locally. Run Phase 1 first to verify Boltz-2 output parsing before committing GPU time to the rest.

| Phase | Dataset | Cas protein | Activity metric | Sign convention | N guides | Source files |
|---|---|---|---|---|---|---|
| 1 | TIGER hap-titration | CasRx (RfxCas13d) | mean LFC | more negative = better | 500 | `af3_v2_metadata.csv` |
| 2 | Konermann CasRx individual | CasRx (RfxCas13d) | mean relative expression (3 reps) | lower = better knockdown | 52 | Konermann xlsx files 1+2 joined |
| 3 | Konermann DjCas13d individual | DjCas13d | mean relative expression (3 reps) | lower = better knockdown | 85 | Konermann xlsx files 1+2 joined |
| 4 | Azimuth V1 | SpCas9 | per-gene flow cytometry LFC | more negative = better | 882 | `V1_data.xlsx` (Human sheet) |

**Optional future phase** — Azimuth V2 (4193 SpCas9 guides, 15 genes, drug survival screen LFC). Excluded for now because the activity metric requires per-gene sign-direction decisions (some genes are pro-survival, others anti-survival in each drug condition), adding interpretive complexity.

**Dropped datasets:**
- *Konermann CasRx survival screen / validation screen*: these large pooled screens do not exist as separate data files in this paper. The only Konermann data files are the individual guide xlsx files used for Phases 2–3.
- *Supplemental_File_1 (Nucleaseq)*: contains only **1 Matched guide per sheet**. The file is a saturation mutagenesis mismatch-tolerance study of a single guide-target pair, not a multi-guide efficiency dataset. Not usable for correlation.

---

### Notes on joining Konermann files (Phases 2–3)

- **File 1** — `[Konermann 2023] Individual guide sequences.xlsx`: 159 rows, columns `name`, `DR sequence`, `spacer sequence`. Guide name prefix encodes effector: `CasRx_*`, `Dj_*`, `Cas7-11_*`. Strip prefix to get the bare guide name for joining.
- **File 2** — `[Konermann 2023] Raw data for individual guide knockdown efficiency and survival rate.xlsx`: 250 rows, `effector` and `guide name` columns with 3-replicate `relative expression` and `survival rate`. Forward-fill the `effector` column (merged cells).
- **Join key**: strip `CasRx_` / `Dj_` prefix from File 1 `name` → match File 2 `guide name` filtered by `effector`.
- **Coverage**: CasRx: 52 guides with both sequence and efficiency (2 NT controls in File 1 have no File 2 match — excluded). DjCas13d: 85 guides with overlap (8 guides in File 2 lack sequences; 8 in File 1 lack efficiency data — excluded).
- **Activity metric**: average 3 replicate columns for `relative expression of target transcript` → `mean_relative_expression`. Lower = better knockdown.

---

### DR sequences for crRNA construction

| Phase | Cas protein | DR sequence (RNA) | Length | Source |
|---|---|---|---|---|
| 1 | CasRx (TIGER) | `AACCCCUACCAACUGGUCGGGGUUUGAAAC` | 30 nt | AF3 pipeline / TIGER paper |
| 2 | CasRx (Konermann) | `CAAGUAAACCCCUACCAACUGGUCGGGGUUUGAAAC` | 36 nt | Konermann xlsx File 1 |
| 3 | DjCas13d | `CAACUACAACCCCGUAAAAAUACGGGGUUCUGAAAC` | 36 nt | Konermann xlsx File 1 |

> The Konermann CasRx 36 nt DR contains the TIGER 30 nt DR as its last 30 nucleotides — both are valid RfxCas13d DRs, but use the DR that matches each specific experiment.

---

### Complex geometry by Cas type

**Cas13 (RNA-targeting — Phases 1–3)**
```
Chain A: Cas protein         (protein)
Chain B: crRNA = DR + spacer (rna)
Chain C: target RNA          (rna) = reverse_complement_rna(spacer)
```

**SpCas9 (DNA-targeting — Phase 4)**  
Requires confirming that Boltz-2 supports `dna` chain type (see Pre-flight Checklist).
```
Chain A: Cas protein (SpCas9, 1368 aa)        (protein)
Chain B: sgRNA = spacer_rna + scaffold         (rna)    96 nt total
Chain C: protospacer + PAM (NGG sense strand)  (dna)    23 nt
```

---
## 3. Protein Sequences

### 3a. CasRx (RfxCas13d)
- **UniProt**: A0A1C5SD84, 967 aa, starts with MAST
- **Already handled** by `fill_casrx_sequence.py` in the AF3 pipeline
- Reuse the same script; point it at the new YAML input directory

### 3b. DjCas13d
- **Source**: `[Konermann 2023] Cas13d ortholog sequences.csv`, row where `Name == "DjCas13d"`
- **Length**: 877 aa
- **Sequence identity to CasRx**: 29.9% — Boltz will treat this as a distinct protein

### 3c. SpCas9 (S. pyogenes)
- **UniProt**: Q99ZW2, ~1368 aa
- Fetch with the same UniProt pattern used for CasRx

### 3d. SpCas9 variants (Phase 6 only)
- SpCas9-eHF1 (enhanced HiFi): Vakulskas et al. 2018 Nat Med
- SpCas9-HyPa: Chen et al. 2017 Nature
- SpCas9-HF1: Kleinstiver et al. 2016 Nature
- These differ from wt SpCas9 by single-digit point mutations — fetch wt sequence and apply mutations manually, or source pre-mutated sequences from supplementary materials of those papers

### 3e. AsCas12a
- **UniProt**: A0Q7Q2 (Acidaminococcus sp. BV3L6)
- ~1307 aa
- Different PAM (5'-TTTV on the non-target strand) and crRNA structure from SpCas9

In [ ]:
"""Read DjCas13d sequence from the local Konermann ortholog CSV."""
import csv

ORTHOLOG_CSV = r"[Konermann 2023] Cas13d ortholog sequences.csv"

dj_seq = None
with open(ORTHOLOG_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if "DjCas13d" in row["Name"]:
            dj_seq = row["Sequence"].strip()
            print(f"Found: {row['Name']}")
            print(f"Length: {len(dj_seq)} aa")
            print(f"First 20 aa: {dj_seq[:20]}")
            break

if dj_seq is None:
    print("ERROR: DjCas13d not found — check Name column values")
    # Print available names for debugging
    with open(ORTHOLOG_CSV, newline="", encoding="utf-8") as f:
        names = [row["Name"] for row in csv.DictReader(f)]
    print("Available names:", names[:10])

---
## 4. Target Sequence Reconstruction

### 4a. Cas13 datasets (Phases 2–4)

The crRNA spacer is complementary to the target mRNA. Given the spacer (5'→3'), the target RNA is its reverse complement in RNA alphabet.

For TIGER (Phase 1), the `target_seq` column in `af3_v2_metadata.csv` already contains pre-computed 23 nt target sequences.

For Konermann screens (Phases 2–4), only the spacer is provided — compute target as shown below.

In [ ]:
"""Reverse complement a RNA spacer to get the target RNA sequence."""

def reverse_complement_rna(seq: str) -> str:
    """Return the reverse complement of an RNA sequence (uses U not T)."""
    seq = seq.upper().replace("T", "U")  # normalise to RNA
    complement = str.maketrans("ACGU", "UGCA")
    return seq.translate(complement)[::-1]


# Examples
test_cases = [
    # (spacer, expected_target)                       # from TIGER af3_v2_metadata as sanity check
    ("GUACGGAGUAAGAGACGUAAGUG", "CACUUACGUCUCUUACUCCGUAC"),
]
for spacer, expected in test_cases:
    result = reverse_complement_rna(spacer)
    match = "OK" if result == expected else f"MISMATCH (got {result})"
    print(f"spacer:   {spacer}")
    print(f"target:   {result}")
    print(f"expected: {expected} [{match}]")
    print()

### 4b. SpCas9 datasets (Phases 5–6)

**V1_data.xlsx**: the `30mer` column = 4 nt context + 20 nt guide + NGG + 3 nt context. Extract positions [4:24] as the protospacer. The DNA target sense strand = that same 20 nt protospacer + NGG.

**V2_data.xlsx**: same 30mer convention.

**Supplemental_File_1.xlsx**: the `Matched` row contains the on-target guide sequence. The target DNA = reverse complement of the guide sequence (in DNA) + PAM. The PAM (NGG for SpCas9, TTTV on NTS for Cas12a) must be appended.

**sgRNA scaffold** (constant, for Chain B):
- SpCas9: `GUUUUAGAGCUAGAAAUAGCAAGUUAAAAUAAGGCUAGUCCGUUAUCAACUUGAAAAAGUGGCACCGAGUCGGUGCUUUU` (76 nt)
- AsCas12a crRNA DR: `AAUUUCUACUAAGUGUAGAU` (20 nt)

In [ ]:
"""Extract protospacer from Azimuth 30mer and build target DNA + sgRNA."""

SPCAS9_SCAFFOLD = "GUUUUAGAGCUAGAAAUAGCAAGUUAAAAUAAGGCUAGUCCGUUAUCAACUUGAAAAAGUGGCACCGAGUCGGUGCUUUU"

def spcas9_chains_from_30mer(thirtmer: str):
    """
    Input : 30mer string from Azimuth V1/V2 (4nt+20nt+NGG+3nt = 30 nt in DNA alphabet)
    Returns:
        sgrna   : 20 nt spacer (DNA→RNA) + scaffold
        target  : 20 nt protospacer + NGG (DNA sense strand)
    """
    thirtmer = thirtmer.upper()
    protospacer_dna = thirtmer[4:24]          # positions 4..23 (0-indexed)
    pam = thirtmer[24:27]                     # NGG at positions 24..26
    spacer_rna = protospacer_dna.replace("T", "U")
    sgrna = spacer_rna + SPCAS9_SCAFFOLD      # 20 nt spacer + 76 nt scaffold
    target_dna = protospacer_dna + pam        # 23 nt (no complement needed for sense strand)
    return sgrna, target_dna


# Test
example_30mer = "AACCGTACGGAGAGTCTTCGAAGGTTGTTC"  # placeholder — replace with real data
sgrna, target = spcas9_chains_from_30mer(example_30mer)
print(f"30mer:   {example_30mer}")
print(f"sgRNA:   {sgrna[:30]}... ({len(sgrna)} nt total)")
print(f"target:  {target} ({len(target)} nt)")

---
## 5. Boltz YAML Input Format

### 5a. Cas13 complex (RNA-targeting — Phases 1–3)

```yaml
version: 1
sequences:
  - protein:
      id: A
      sequence: "MAST..."              # CasRx (967 aa) or DjCas13d (877 aa)
  - rna:
      id: B
      sequence: "<DR><spacer>"         # DR + spacer, all RNA alphabet (U not T)
  - rna:
      id: C
      sequence: "<reverse_complement_of_spacer>"   # target RNA
```

### 5b. SpCas9 complex (DNA-targeting — Phase 4)

```yaml
version: 1
sequences:
  - protein:
      id: A
      sequence: "MDKKYSIGL..."         # SpCas9 (1368 aa, UniProt Q99ZW2)
  - rna:
      id: B
      sequence: "<spacer_rna><scaffold>"   # 20 nt spacer (U) + 76 nt scaffold = 96 nt
  - dna:
      id: C
      sequence: "<protospacer>NGG"         # 23 nt sense strand (DNA alphabet, T not U)
```

> ⚠️  **Verify** that Boltz-2 supports the `dna` chain type before generating Phase 4 inputs. Run `boltz predict --help` on NSCC and check the [Boltz-2 GitHub README](https://github.com/jwohlwend/boltz).

---

### 5c. DR sequences by dataset

| Phase | Protein | DR sequence (RNA alphabet, U not T) | Length |
|---|---|---|---|
| 1 — TIGER | CasRx | `AACCCCUACCAACUGGUCGGGGUUUGAAAC` | 30 nt |
| 2 — Konermann CasRx | CasRx | `CAAGUAAACCCCUACCAACUGGUCGGGGUUUGAAAC` | 36 nt |
| 3 — Konermann DjCas13d | DjCas13d | `CAACUACAACCCCGUAAAAAUACGGGGUUCUGAAAC` | 36 nt |

> **DR disambiguation**: The Konermann CasRx 36 nt DR (`CAAGUA...`) contains the TIGER 30 nt DR as its last 30 nucleotides. They share the same core functional sequence. Use the DR that matches the actual experimental protocol for each dataset — confirmed by checking which sequence appears in the respective xlsx file.
>
> **DjCas13d DR** is converted from the DNA sequence in `[Konermann 2023] Individual guide sequences.xlsx` (`CAACTACAACCCCGTAAAAATACGGGGTTCTGAAAC`) by replacing T→U.

---

### 5d. SpCas9 sgRNA scaffold and 30mer format (Phase 4)

**sgRNA scaffold** (76 nt, constant for all SpCas9 guides):
```
GUUUUAGAGCUAGAAAUAGCAAGUUAAAAUAAGGCUAGUCCGUUAUCAACUUGAAAAAGUGGCACCGAGUCGGUGCUUUU
```

**Azimuth V1 30mer format** (34 nt: 4 nt context + 20 nt protospacer + NGG + 7 nt context):
- Protospacer = `thirtmer[4:24]`
- PAM = `thirtmer[24:27]` (NGG)
- sgRNA Chain B = `protospacer.replace("T","U") + scaffold`
- Target Chain C = `protospacer + PAM` (DNA, sense strand only)

In [ ]:
"""Generate a single Boltz YAML input for a Cas13 complex."""
import os
import yaml   # pip install pyyaml


DR_SEQUENCES = {
    "CasRx":    "AACCCCUACCAACUGGUCGGGGUUUGAAAC",      # 30 nt
    "DjCas13d": "CAACUACAACCCCGUAAAAAUACGGGGTTCUGAAAC", # 36 nt — confirm against PDB 6IV9
}


def make_cas13_yaml(
    job_name: str,
    protein_seq: str,
    cas_type: str,       # "CasRx" or "DjCas13d"
    spacer: str,         # spacer only, without DR
    output_dir: str,
) -> str:
    """Write a Boltz YAML for a Cas13–crRNA–target complex. Returns the file path."""
    dr = DR_SEQUENCES[cas_type]
    crrna = dr + spacer.upper().replace("T", "U")
    target = reverse_complement_rna(spacer)

    data = {
        "version": 1,
        "sequences": [
            {"protein": {"id": "A", "sequence": protein_seq}},
            {"rna":     {"id": "B", "sequence": crrna}},
            {"rna":     {"id": "C", "sequence": target}},
        ],
    }

    os.makedirs(output_dir, exist_ok=True)
    fpath = os.path.join(output_dir, f"{job_name}.yaml")
    with open(fpath, "w") as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)

    return fpath


# Quick sanity check (replace protein_seq with actual sequence on NSCC)
path = make_cas13_yaml(
    job_name="test_casrx_guide",
    protein_seq="MASTPLACEHOLDER",  # real seq fetched in prep step
    cas_type="CasRx",
    spacer="GUACGGAGUAAGAGACGUAAGUG",
    output_dir="./boltz_inputs_test",
)
print(f"Written: {path}")
with open(path) as f:
    print(f.read())

In [ ]:
"""Generate Boltz YAML inputs for ALL Cas13 datasets from their metadata CSVs.

Run this locally (or in a PBS prep job on NSCC) before submitting inference.
Expects the following metadata CSV columns:
  - job_name   : unique identifier used as filename and output folder name
  - guide_seq  : spacer sequence (RNA or DNA alphabet, normalised internally)
  - [activity] : activity column (not used here, just preserved in metadata)

For TIGER: reuse af3_v2_metadata.csv (already has job_name + guide_seq + target_seq).
"""
import csv

DATASETS = [
    {
        "name":         "tiger",
        "metadata_csv": "alphafold3/my_parts/af3_v2_metadata.csv",
        "cas_type":     "CasRx",
        "spacer_col":   "guide_seq",
        "job_col":      "job_name",
        "output_dir":   "boltz_inputs/tiger",
    },
    {
        "name":         "konermann_survival",
        "metadata_csv": "boltz_konermann_survival_metadata.csv",  # to be generated
        "cas_type":     "CasRx",
        "spacer_col":   "guide_seq",
        "job_col":      "job_name",
        "output_dir":   "boltz_inputs/konermann_survival",
    },
    {
        "name":         "konermann_validation",
        "metadata_csv": "boltz_konermann_validation_metadata.csv",  # to be generated
        "cas_type":     "CasRx",
        "spacer_col":   "guide_seq",
        "job_col":      "job_name",
        "output_dir":   "boltz_inputs/konermann_validation",
    },
    {
        "name":         "konermann_djcas13d",
        "metadata_csv": "boltz_konermann_djcas13d_metadata.csv",  # to be generated
        "cas_type":     "DjCas13d",
        "spacer_col":   "guide_seq",
        "job_col":      "job_name",
        "output_dir":   "boltz_inputs/konermann_djcas13d",
    },
]

# Protein sequences — filled in during prep step (see Section 3)
PROTEIN_SEQUENCES = {
    "CasRx":    "FETCH_FROM_UNIPROT_A0A1C5SD84",   # fill_casrx_sequence.py handles this
    "DjCas13d": "FETCH_FROM_ORTHOLOG_CSV",          # read above in Section 3a cell
}


def generate_dataset_yamls(dataset_cfg: dict) -> int:
    """Generate YAML files for one dataset. Returns number of files written."""
    cas_type = dataset_cfg["cas_type"]
    protein_seq = PROTEIN_SEQUENCES[cas_type]
    count = 0

    with open(dataset_cfg["metadata_csv"], newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            job_name = row[dataset_cfg["job_col"]]
            spacer   = row[dataset_cfg["spacer_col"]]
            make_cas13_yaml(
                job_name   = job_name,
                protein_seq = protein_seq,
                cas_type   = cas_type,
                spacer     = spacer,
                output_dir = dataset_cfg["output_dir"],
            )
            count += 1

    print(f"{dataset_cfg['name']}: {count} YAML files written → {dataset_cfg['output_dir']}")
    return count


# Uncomment to run (protein sequences must be filled first):
# for ds in DATASETS:
#     generate_dataset_yamls(ds)

---
## 6. PBS Scripts and NSCC Setup

### Full setup sequence

```
[Windows]  Download weight files from HuggingFace
[Windows]  scp -O weight files + PBS scripts → NSCC scratch
[NSCC]     mkdir boltz_cache/ and logs/ before scp
[NSCC]     qsub 00_install_boltz.pbs   ← creates conda env + pip install + extracts mols.tar
[NSCC]     qsub 01_test_boltz.pbs      ← verifies GPU + prints confidence JSON key names
[NSCC]     qsub 02_boltz_inference.pbs ← PBS array, one task per YAML
[NSCC]     qsub 03_analyze.pbs         ← correlation analysis
```

---

### Step 1 — Download weight files (Windows, browser or PowerShell)

Download these two files from HuggingFace — you may need a free HuggingFace account:

```
https://huggingface.co/boltz-community/boltz-2/resolve/main/boltz2_conf.ckpt
https://huggingface.co/boltz-community/boltz-2/resolve/main/mols.tar
```

Do **not** download `boltz2_aff.ckpt` — it is affinity prediction only and not needed here.

---

### Step 2 — Create directories on NSCC (SSH, login node)

`scp` cannot create remote directories — these must exist before transferring files:

```bash
mkdir -p /scratch/users/ntu/hzhang08/boltz_cache
mkdir -p /scratch/users/ntu/hzhang08/logs
```

---

### Step 3 — Transfer files via scp -O (Windows PowerShell)

The `-O` flag forces the legacy SCP protocol, avoiding SFTP compatibility issues on NSCC.

```powershell
# Weight files (large — transfer first, takes time)
scp -O "C:\Users\holly\Documents\Documents\URECA_Code\boltz2_conf.ckpt" hzhang08@aspire2antu.nscc.sg:/scratch/users/ntu/hzhang08/boltz_cache/
scp -O "C:\Users\holly\Documents\Documents\URECA_Code\mols.tar"          hzhang08@aspire2antu.nscc.sg:/scratch/users/ntu/hzhang08/boltz_cache/

# PBS scripts
scp -O "C:\Users\holly\Documents\Documents\URECA_Code\boltz\00_install_boltz.pbs"    hzhang08@aspire2.nscc.sg:/scratch/users/ntu/hzhang08/
scp -O "C:\Users\holly\Documents\Documents\URECA_Code\boltz\01_test_boltz.pbs"       hzhang08@aspire2.nscc.sg:/scratch/users/ntu/hzhang08/
scp -O "C:\Users\holly\Documents\Documents\URECA_Code\boltz\02_fix_torch_cuda.pbs"   hzhang08@aspire2.nscc.sg:/scratch/users/ntu/hzhang08/
```

---

### Step 4 — Submit install job (NSCC login node)

The conda environment is **created inside this PBS job** — do not try to create it manually on the login node.

```bash
qsub /scratch/users/ntu/hzhang08/00_install_boltz.pbs
```

This CPU job (~30–60 min):
1. Checks `boltz2_conf.ckpt` and `mols.tar` exist — exits with a clear error if either is missing
2. Creates conda env at `$SCRATCH/envs/boltz` (not home — no space there)
3. `pip install boltz`
4. Extracts `mols.tar` → `mols/` in the cache directory (Boltz-2 needs the extracted directory, not the tar)

Check the log when done:
```bash
cat /scratch/users/ntu/hzhang08/logs/install_boltz.out
```

---

### Step 5 — GPU test (NSCC login node)

```bash
qsub /scratch/users/ntu/hzhang08/01_test_boltz.pbs
```

Check log when done:
```bash
cat /scratch/users/ntu/hzhang08/logs/boltz_test.out
```

Two things to verify:
1. `CUDA available: True` — if False, edit `CUDA_TAG` in `02_fix_torch_cuda.pbs`, scp it over, submit it, then resubmit `01_test_boltz.pbs`
2. The confidence JSON printed at the end — copy those key names and update `parse_boltz_confidence()` in Section 7 if they differ

---

### Key path reference

| Resource | Path on NSCC scratch |
|---|---|
| Conda env | `/scratch/users/ntu/hzhang08/envs/boltz` |
| Weight checkpoint | `/scratch/users/ntu/hzhang08/boltz_cache/boltz2_conf.ckpt` |
| Molecular data | `/scratch/users/ntu/hzhang08/boltz_cache/mols/` (extracted from `mols.tar`) |
| YAML inputs | `/scratch/users/ntu/hzhang08/boltz_inputs/<dataset>/` |
| Inference outputs | `/scratch/users/ntu/hzhang08/boltz_outputs/<dataset>/` |
| Analysis results | `/scratch/users/ntu/hzhang08/results/boltz_<dataset>/` |

---

### 02_boltz_inference.pbs (GPU array)

```bash
#!/bin/bash
#PBS -N boltz_inf
#PBS -l select=1:ncpus=8:mem=64gb:ngpus=1
#PBS -l walltime=04:00:00
#PBS -J 0-499
#PBS -o /scratch/users/ntu/hzhang08/logs/boltz_^array_index^.out
#PBS -e /scratch/users/ntu/hzhang08/logs/boltz_^array_index^.err

SCRATCH="/scratch/users/ntu/hzhang08"
ENV_PATH="$SCRATCH/envs/boltz"
BOLTZ_CACHE="$SCRATCH/boltz_cache"

# ── Edit these two per dataset ────────────────────────────────
YAML_DIR="$SCRATCH/boltz_inputs/tiger"
OUTPUT_DIR="$SCRATCH/boltz_outputs/tiger"
# ─────────────────────────────────────────────────────────────

source ~/miniconda3/bin/activate "$ENV_PATH"
mkdir -p "$OUTPUT_DIR"

YAML_FILES=($YAML_DIR/*.yaml)
TOTAL=${#YAML_FILES[@]}

if [ $PBS_ARRAY_INDEX -ge $TOTAL ]; then
    echo "Index $PBS_ARRAY_INDEX >= $TOTAL — nothing to do"
    exit 0
fi

THIS_YAML=${YAML_FILES[$PBS_ARRAY_INDEX]}
echo "Processing: $(basename $THIS_YAML)  Start: $(date)"

boltz predict "$THIS_YAML" \
    --out_dir "$OUTPUT_DIR" \
    --cache "$BOLTZ_CACHE" \
    --recycling_steps 3 \
    --diffusion_samples 5 \
    --use_msa_server false

echo "Done: $(date)"
```

> **Array size**: set `-J 0-N` where N = number of YAML files minus 1.  
> **Walltime**: Boltz-2 on an A100 — expect 5–20 min per complex with 5 diffusion samples.

---

### 03_analyze.pbs (CPU job)

```bash
#!/bin/bash
#PBS -N boltz_analyze
#PBS -l select=1:ncpus=4:mem=16gb
#PBS -l walltime=01:00:00
#PBS -o /scratch/users/ntu/hzhang08/logs/boltz_analyze.out
#PBS -e /scratch/users/ntu/hzhang08/logs/boltz_analyze.err

SCRATCH="/scratch/users/ntu/hzhang08"
ENV_PATH="$SCRATCH/envs/boltz"

source ~/miniconda3/bin/activate "$ENV_PATH"

# ── Edit per dataset ──────────────────────────────────────────
DATASET="tiger"
ACTIVITY_COL="mean_lfc"
BASELINE_COLS="hybrid_mfe_1_23,log_unpaired"
# ─────────────────────────────────────────────────────────────

python3 $HOME/analyze_boltz_results.py \
    --boltz_dir    "$SCRATCH/boltz_outputs/$DATASET" \
    --metadata     "$SCRATCH/boltz_inputs/${DATASET}_metadata.csv" \
    --activity_col "$ACTIVITY_COL" \
    --baseline_cols "$BASELINE_COLS" \
    --output_dir   "$SCRATCH/results/boltz_$DATASET"
```

---
## 7. Parsing Boltz Outputs

Boltz output directory structure (one subfolder per job):

```
boltz_outputs/tiger/
  predictions/
    casrx_guide_0_004/
      casrx_guide_0_004_model_0.cif
      casrx_guide_0_004_confidence_model_0.json   ← confidence scores
      casrx_guide_0_004_model_1.cif               ← seed 2
      casrx_guide_0_004_confidence_model_1.json
      ...                                          ← seeds 3–5
```

**Boltz confidence JSON keys** (verify field names on NSCC before finalising the parser):

| Key | Meaning | AF3 equivalent |
|---|---|---|
| `ptm` | Overall complex pTM | `ptm` |
| `iptm` | Interface pTM | `iptm` |
| `complex_plddt` | Mean pLDDT over all chains (0–1) | `mean_plddt` / 100 |
| `plddt` | Per-residue pLDDT list (0–1) | `atom_plddts` / 100 |
| `chains` | Per-chain breakdown dict | `chain_ptm`, `chain_iptm` |
| `gpde` | Global predicted distance error | no direct equivalent |

> **Scale**: Boltz pLDDT is 0–1. Multiply by 100 before comparison with AF3 values, or normalise AF3 to 0–1 at reporting time.

In [ ]:
"""analyze_boltz_results.py — parse Boltz confidence JSONs and correlate with activity.

Mirrors analyze_af3_results.py but handles Boltz output structure and key names.
Run from 03_analyze.pbs on NSCC.
"""
import os
import json
import glob
import argparse
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from numpy.linalg import lstsq

warnings.filterwarnings("ignore")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

# Chain assignments (must match the YAML order)
CHAIN_PROTEIN = "A"
CHAIN_CRRNA   = "B"
CHAIN_TARGET  = "C"

# CasRx-specific (adjust for other datasets)
DR_LENGTH     = 30   # for CasRx; 36 for DjCas13d
SPACER_LENGTH = 23   # for TIGER; 30 for Konermann survival


def parse_boltz_confidence(json_path: str) -> dict:
    """
    Parse a single Boltz confidence_model_N.json file.

    IMPORTANT: Boltz pLDDT is 0–1. We multiply by 100 here to match AF3 scale.
    Verify field names on NSCC — Boltz may update its output schema.
    """
    with open(json_path) as f:
        conf = json.load(f)

    out = {
        "ptm":  conf.get("ptm",  np.nan),
        "iptm": conf.get("iptm", np.nan),
    }

    # Complex-level pLDDT (Boltz 0–1 → multiply to 0–100)
    cplddt = conf.get("complex_plddt", np.nan)
    out["complex_plddt"] = cplddt * 100 if not np.isnan(cplddt) else np.nan

    # Per-residue pLDDT (list, 0–1)
    plddt_list = conf.get("plddt", [])
    if plddt_list:
        plddt = np.array(plddt_list) * 100  # rescale to 0–100
        out["mean_plddt"] = float(np.mean(plddt))

        # Chain-level means — requires knowing residue boundaries
        # Boltz may provide per-chain breakdown under "chains" key
        chains = conf.get("chains", {})
        for chain_id, label in [(CHAIN_PROTEIN, "plddt_A"),
                                 (CHAIN_CRRNA,   "plddt_B"),
                                 (CHAIN_TARGET,  "plddt_C")]:
            chain_data = chains.get(chain_id, {})
            chain_plddt = chain_data.get("plddt", np.nan)
            if not np.isnan(chain_plddt):
                out[label] = chain_plddt * 100

    # Chain-pair ipTM (if available)
    chain_iptm = conf.get("chain_pair_iptm", conf.get("pair_chains_iptm", {}))
    if chain_iptm:
        out["iptm_AB"] = chain_iptm.get(CHAIN_PROTEIN, {}).get(CHAIN_CRRNA,  np.nan)
        out["iptm_AC"] = chain_iptm.get(CHAIN_PROTEIN, {}).get(CHAIN_TARGET, np.nan)
        out["iptm_BC"] = chain_iptm.get(CHAIN_CRRNA,  {}).get(CHAIN_TARGET, np.nan)

    # Global predicted distance error (lower = more confident)
    out["gpde"] = conf.get("gpde", np.nan)

    return out


def parse_one_job(job_dir: str) -> dict:
    """Aggregate confidence across all seeds for one guide."""
    job_name = os.path.basename(job_dir)
    conf_files = sorted(glob.glob(os.path.join(job_dir, "*confidence_model_*.json")))

    if not conf_files:
        print(f"  WARNING: No confidence JSONs in {job_name}")
        return {"job_name": job_name}

    seed_records = []
    for cf in conf_files:
        try:
            seed_records.append(parse_boltz_confidence(cf))
        except Exception as e:
            print(f"  WARNING: {os.path.basename(cf)}: {e}")

    if not seed_records:
        return {"job_name": job_name}

    # Average across seeds (same aggregation strategy as AF3 experiment)
    df = pd.DataFrame(seed_records)
    result = {"job_name": job_name, "n_seeds": len(seed_records)}
    for col in df.columns:
        result[col] = float(df[col].mean())

    return result


def parse_all_jobs(boltz_dir: str) -> pd.DataFrame:
    """Walk boltz_dir/predictions/ and parse every job subdirectory."""
    pred_dir = os.path.join(boltz_dir, "predictions")
    if not os.path.isdir(pred_dir):
        pred_dir = boltz_dir  # fallback: flat structure

    job_dirs = sorted([
        d for d in glob.glob(os.path.join(pred_dir, "*"))
        if os.path.isdir(d)
    ])

    if not job_dirs:
        print(f"ERROR: No job directories found under {pred_dir}")
        return pd.DataFrame()

    print(f"Found {len(job_dirs)} job directories")
    records = []
    for d in job_dirs:
        try:
            records.append(parse_one_job(d))
        except Exception as e:
            print(f"  ERROR: {os.path.basename(d)}: {e}")

    return pd.DataFrame(records)

---
## 8. Statistical Analysis

The three-stage analysis from the AF3 experiment carries over unchanged. The functions below are dataset-agnostic — pass the appropriate `activity_col` for each dataset.

| Phase | Dataset | `activity_col` | Direction | Baseline cols (partial corr) |
|---|---|---|---|---|
| 1 | TIGER CasRx | `mean_lfc` | more negative = better | `hybrid_mfe_1_23`, `log_unpaired` |
| 2 | Konermann CasRx | `mean_relative_expression` | lower = better knockdown | guide GC content |
| 3 | Konermann DjCas13d | `mean_relative_expression` | lower = better knockdown | guide GC content |
| 4 | Azimuth V1 SpCas9 | `mean_lfc` (averaged across available cell-line columns per guide) | more negative = better | guide GC content, `percent_peptide` |

**Azimuth V1 activity construction**: each guide has measurements in only the cell lines matching its target gene (CD13 guides → NB4_CD13 + TF1_CD13; CD15 → MOLM13_CD15; CD33 → MOLM13_CD33 + NB4_CD33 + TF1_CD33). Average all non-NaN LFC columns per guide → `mean_lfc`. Standardise within each gene before pooling across genes for the combined analysis.

**Bonferroni correction**: ~10 Boltz metrics × 4 datasets = 40 tests → corrected α = 0.05/40 = **0.00125**.

**Cross-dataset summary**: after all phases complete, compile one table of Spearman r(ipTM, activity) per phase to test whether Cas13 (RNA-targeting, flexible target) shows stronger correlation than SpCas9 (DNA-targeting, rigid PAM-constrained complex).

In [ ]:
"""Statistical analysis functions — dataset-agnostic. Mirrors analyze_af3_results.py."""
from scipy import stats
from numpy.linalg import lstsq
import numpy as np
import pandas as pd


BOLTZ_METRICS = [
    "ptm", "iptm", "complex_plddt", "mean_plddt",
    "plddt_A", "plddt_B", "plddt_C",
    "iptm_AB", "iptm_AC", "iptm_BC",
    "gpde",
]


def spearman_pearson(x, y):
    valid = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valid) < 10:
        return np.nan, np.nan, np.nan, np.nan
    sp_r, sp_p = stats.spearmanr(valid["x"], valid["y"])
    pe_r, pe_p = stats.pearsonr(valid["x"],  valid["y"])
    return float(sp_r), float(sp_p), float(pe_r), float(pe_p)


def partial_correlation(df, metric, baseline_col, outcome):
    """Spearman partial correlation: metric vs outcome controlling for baseline."""
    if isinstance(baseline_col, str):
        baseline_cols = [baseline_col]
    else:
        baseline_cols = list(baseline_col)

    valid = df[[metric, outcome] + baseline_cols].dropna()
    if len(valid) < 20:
        return np.nan, np.nan, len(valid)

    X = np.column_stack([valid[b].values for b in baseline_cols] + [np.ones(len(valid))])
    resid_m = valid[metric].values  - X @ lstsq(X, valid[metric].values,  rcond=None)[0]
    resid_o = valid[outcome].values - X @ lstsq(X, valid[outcome].values, rcond=None)[0]
    r, p = stats.spearmanr(resid_m, resid_o)
    return float(r), float(p), len(valid)


def run_full_analysis(
    boltz_dir:    str,
    metadata_csv: str,
    activity_col: str,
    baseline_cols: list,
    output_dir:   str,
    alpha_bonf:   float = 0.00083,   # 0.05 / 60 tests across all datasets
):
    """Full pipeline: parse → merge → correlate → save. One call per dataset."""
    os.makedirs(output_dir, exist_ok=True)

    # 1. Parse outputs
    boltz_df = parse_all_jobs(boltz_dir)
    if boltz_df.empty:
        print("No Boltz outputs found."); return

    # 2. Merge with metadata
    meta = pd.read_csv(metadata_csv)
    merged = boltz_df.merge(meta, on="job_name", how="inner")
    print(f"Matched {len(merged)} / {len(boltz_df)} jobs to metadata")

    merged.to_csv(os.path.join(output_dir, "boltz_merged.csv"), index=False)

    # 3. Stage 1: raw correlations
    records = []
    print(f"\n{'Metric':<22} {'Spearman r':>12} {'p-value':>12} {'n':>6} {'sig':>5}")
    print("-" * 60)
    for col in BOLTZ_METRICS + baseline_cols:
        if col not in merged.columns: continue
        sp_r, sp_p, pe_r, _ = spearman_pearson(merged[col], merged[activity_col])
        if np.isnan(sp_r): continue
        n = merged[[col, activity_col]].dropna().__len__()
        sig = "***" if sp_p < alpha_bonf else ("*" if sp_p < 0.05 else "")
        tag = " [base]" if col in baseline_cols else ""
        print(f"  {col+tag:<20} {sp_r:>12.4f} {sp_p:>12.4e} {n:>6} {sig:>5}")
        records.append({"metric": col, "spearman_r": sp_r, "spearman_p": sp_p,
                        "pearson_r": pe_r, "n": n, "is_baseline": col in baseline_cols})

    corr_df = pd.DataFrame(records)
    corr_df.to_csv(os.path.join(output_dir, "correlations.csv"), index=False)

    # 4. Stage 2: partial correlations
    if baseline_cols:
        print(f"\n{'Metric':<18} {'Baseline':<20} {'Partial r':>10} {'p':>12} {'n':>6}")
        print("-" * 70)
        partial_records = []
        for m in ["iptm", "ptm", "complex_plddt", "iptm_AB", "iptm_BC"]:
            if m not in merged.columns: continue
            for b in baseline_cols:
                if b not in merged.columns: continue
                r, p, n = partial_correlation(merged, m, b, activity_col)
                if np.isnan(r): continue
                sig = "***" if p < alpha_bonf else ("*" if p < 0.05 else "")
                print(f"  {m:<16} {b:<20} {r:>10.4f} {p:>12.4e} {n:>6} {sig}")
                partial_records.append({"metric": m, "baseline": b, "partial_r": r, "partial_p": p, "n": n})
        pd.DataFrame(partial_records).to_csv(
            os.path.join(output_dir, "partial_correlations.csv"), index=False)

    print(f"\nAll results saved to: {output_dir}")
    return corr_df


# Example invocation (run in 03_analyze.pbs):
# run_full_analysis(
#     boltz_dir     = "/scratch/users/ntu/hzhang08/boltz_outputs/tiger",
#     metadata_csv  = "/scratch/users/ntu/hzhang08/boltz_inputs/tiger/tiger_metadata.csv",
#     activity_col  = "mean_lfc",
#     baseline_cols = ["hybrid_mfe_1_23", "log_unpaired"],
#     output_dir    = "/scratch/users/ntu/hzhang08/results/boltz_tiger",
# )

---
## 9. Pre-flight Checklist

### One-time NSCC setup (in order)

- [ ] **SSH in** → `mkdir -p /scratch/users/ntu/hzhang08/boltz_cache` and `mkdir -p /scratch/users/ntu/hzhang08/logs`
- [ ] **Download weights locally** (Windows): `boltz2_conf.ckpt` (2.29 GB) and `mols.tar` (1.86 GB) from `huggingface.co/boltz-community/boltz-2`. Skip `boltz2_aff.ckpt` — affinity only.
- [ ] **scp -O** weight files to `$SCRATCH/boltz_cache/` and PBS scripts to `$SCRATCH/`
- [ ] **`qsub 00_install_boltz.pbs`** — creates conda env in scratch, pip installs boltz, extracts `mols.tar` → `mols/`. Check log for any missing file errors.
- [ ] **`qsub 01_test_boltz.pbs`** — runs minimal CasRx complex on A100. Verify `CUDA available: True` in log.
- [ ] **Copy confidence JSON key names** from `01_test_boltz.pbs` log → update `parse_boltz_confidence()` in Section 7 if they differ from the table there
- [ ] If `CUDA available: False`: edit `CUDA_TAG` in `02_fix_torch_cuda.pbs`, scp it, submit it, then rerun `01_test_boltz.pbs`
- [ ] Confirm Boltz-2 supports `dna` chain type (check `boltz predict --help` output from `01_test_boltz.pbs` log) — required for Phase 4

---

### Per-dataset preparation (before generating YAML inputs)

**Phase 1 — TIGER CasRx (500 guides)**
- [ ] Reuse `af3_v2_metadata.csv` directly — 500 guides already have `guide_seq` (23 nt spacer) and `target_seq`. No new sampling needed.
- [ ] DR: `AACCCCUACCAACUGGUCGGGGUUUGAAAC` (30 nt) — same as used in the AF3 experiment.

**Phase 2 — Konermann CasRx (52 guides)**
- [ ] Join `[Konermann 2023] Individual guide sequences.xlsx` (filter `name` starts with `CasRx_`, strip prefix) with `[Konermann 2023] Raw data...xlsx` (filter `effector == "CasRx"`) on guide name.
- [ ] Average 3 replicate `relative expression` columns → `mean_relative_expression`.
- [ ] DR: `CAAGUAAACCCCUACCAACUGGUCGGGGUUUGAAAC` (36 nt, from xlsx file; T→U converted).
- [ ] Spacer: 30 nt (from `spacer sequence` column).
- [ ] Exclude NT1 and NT2 (non-targeting controls — no efficiency data in raw file).

**Phase 3 — Konermann DjCas13d (85 guides)**
- [ ] Same join procedure, filter `Dj_` prefix / `effector == "DjCas13d"`. 85 guides have both sequence and efficiency data.
- [ ] Activity: `mean_relative_expression` (same as Phase 2).
- [ ] DR: `CAACUACAACCCCGUAAAAAUACGGGGUUCUGAAAC` (36 nt, T→U converted).
- [ ] Note: 8 guides in raw data have no sequence entry; 8 in sequence file have no raw efficiency row — both groups excluded.

**Phase 4 — Azimuth V1 SpCas9 (882 guides)**
- [ ] Use `V1_data.xlsx` Human sheet. No join needed — all in one file.
- [ ] Protospacer: `30mer[4:24]` (20 nt); PAM: `30mer[24:27]` (NGG).
- [ ] sgRNA Chain B = `protospacer.replace("T","U") + scaffold` (96 nt total).
- [ ] Target Chain C = `protospacer + PAM` (23 nt, DNA alphabet).
- [ ] Activity: average all non-NaN LFC columns per guide → `mean_lfc`. Standardise within each gene (CD13, CD15, CD33) before combining.
- [ ] Confirm Boltz-2 `dna` chain type is supported before generating YAMLs.
- [ ] SpCas9 sequence: UniProt Q99ZW2 (~1368 aa).

---

### After Phase 1 completes — before proceeding to Phases 2–4

- [ ] `boltz_merged.csv` contains all 500 rows
- [ ] pLDDT values confirm 0–1 range before ×100 rescaling
- [ ] Compare Spearman r(ipTM, mean_lfc) between Boltz-2 and AF3 on the same 500 guides

---

### Files to produce locally and transfer to NSCC

| File | Content | Where on NSCC |
|---|---|---|
| `boltz2_conf.ckpt` | Boltz-2 structure + confidence weights | `$SCRATCH/boltz_cache/` |
| `mols.tar` | Molecular data archive | `$SCRATCH/boltz_cache/` (extracted → `mols/`) |
| `00_install_boltz.pbs` | Install job | `$SCRATCH/` |
| `01_test_boltz.pbs` | GPU test job | `$SCRATCH/` |
| `02_fix_torch_cuda.pbs` | CUDA fix (if needed) | `$SCRATCH/` |
| `boltz_inputs/tiger/*.yaml` | 500 YAML files (Phase 1) | `$SCRATCH/boltz_inputs/tiger/` |
| `boltz_inputs/tiger/tiger_metadata.csv` | Guide metadata + LFC | `$SCRATCH/boltz_inputs/tiger/` |
| `boltz_inputs/konermann_casrx/*.yaml` | 52 YAML files (Phase 2) | `$SCRATCH/boltz_inputs/konermann_casrx/` |
| `boltz_inputs/konermann_djcas13d/*.yaml` | 85 YAML files (Phase 3) | `$SCRATCH/boltz_inputs/konermann_djcas13d/` |
| `boltz_inputs/azimuth_v1/*.yaml` | 882 YAML files (Phase 4) | `$SCRATCH/boltz_inputs/azimuth_v1/` |
| `02_boltz_inference.pbs` | GPU array inference job | `$SCRATCH/` |
| `03_analyze.pbs` + `analyze_boltz_results.py` | Analysis job + script | `$SCRATCH/` and `$HOME/` |